# State of Data — EDA Regional: Comparação entre Regiões do Brasil (2021–2024)
Análise exploratória comparando perfil dos profissionais de dados por região, com recorte de gênero.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Constantes ────────────────────────────────────────────────────────────
CORES_GENERO  = {'Masculino': '#1E90FF', 'Feminino': '#FF7F50'}
CORES_REGIAO  = {
    'Sudeste':      '#2563EB',
    'Sul':          '#16A34A',
    'Nordeste':     '#EA580C',
    'Centro-Oeste': '#9333EA',
    'Norte':        '#CA8A04',
}
REGIOES       = ['Sudeste', 'Sul', 'Nordeste', 'Centro-Oeste', 'Norte']
ANOS          = ['2021', '2022', '2023', '2024']
TEMPLATE      = 'plotly_white'

FAIXA_SALARIAL_ORDEM = [
    'R$1k-2k','R$2k-3k','R$3k-4k','R$4k-6k','R$6k-8k',
    'R$8k-12k','R$12k-16k','R$16k-20k','R$20k-25k',
    'R$25k-30k','R$30k-40k','R$40k+',
]
NIVEL_ENSINO_ORDEM = [
    'Não tenho graduação formal','Estudante de Graduação',
    'Graduação/Bacharelado','Especialização Lato Sensu','Mestrado','Doutorado ou Phd',
]
NIVEL_SENIORIDADE_ORDEM = ['Júnior', 'Pleno', 'Sênior', 'Gestor']

# ── Carrega df tratado ────────────────────────────────────────────────────
df = pd.read_excel('df_full.xlsx')
df['ano_pesquisa']   = df['ano_pesquisa'].astype(str)
df['faixa_salarial'] = pd.Categorical(df['faixa_salarial'], categories=FAIXA_SALARIAL_ORDEM, ordered=True)
df['nivel_ensino']   = pd.Categorical(df['nivel_ensino'],   categories=NIVEL_ENSINO_ORDEM,   ordered=True)
df['nivel']          = pd.Categorical(df['nivel'],          categories=NIVEL_SENIORIDADE_ORDEM, ordered=True)

print(f'Linhas brutas: {len(df):,}')

Linhas brutas: 17,295


## 1. Padronização da coluna `regiao`

In [2]:
# Inspeciona valores únicos antes de padronizar
print('Valores únicos em regiao:')
print(df['regiao'].value_counts(dropna=False).to_string())

Valores únicos em regiao:
regiao
Sudeste         10542
Sul              3044
Nordeste         1971
Centro-oeste     1071
NaN               360
Norte             256
Exterior           51


In [3]:
# Mapeamento de padronização — ajuste se necessário após inspeção acima
REGIAO_MAP = {
    # variações com acento / capitalização / abreviação
    'Centro_Oeste':  'Centro-Oeste',
    'Centro Oeste':  'Centro-Oeste',
    'centro-oeste':  'Centro-Oeste',
    'nordeste':      'Nordeste',
    'norte':         'Norte',
    'sudeste':       'Sudeste',
    'sul':           'Sul',
    # valores que não representam região brasileira → descarta
    'Exterior':      None,
    'Prefiro não informar': None,
}

df['regiao'] = df['regiao'].replace(REGIAO_MAP)

# Mantém só as 5 regiões canônicas
df_reg = df[df['regiao'].isin(REGIOES)].copy()
df_reg['regiao'] = pd.Categorical(df_reg['regiao'], categories=REGIOES, ordered=False)

print(f'Linhas após filtro regional: {len(df_reg):,}')
print()
print(df_reg.groupby(['ano_pesquisa', 'regiao'], observed=True).size().unstack(fill_value=0))

Linhas após filtro regional: 15,813

regiao        Sudeste   Sul  Nordeste  Norte
ano_pesquisa                                
2021             1663   398       300     36
2022             2618   662       558     75
2023             3147   958       608     82
2024             3114  1026       505     63


## Funções auxiliares

In [4]:
def legenda_unica(fig):
    """Remove entradas duplicadas de legenda."""
    vistos = set()
    for trace in fig.data:
        if trace.name in vistos:
            trace.showlegend = False
        else:
            vistos.add(trace.name)
            trace.showlegend = True
    return fig


def layout_padrao(fig, titulo, height=500, **kwargs):
    fig.update_layout(
        title=dict(text=titulo, font=dict(size=15)),
        template=TEMPLATE,
        height=height,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        **kwargs
    )
    return fig


def pct_feminino_por_grupo(df, col_grupo):
    """
    Retorna % feminino dentro de cada valor de col_grupo.
    Útil para comparar representatividade entre regiões / anos.
    """
    ct = df.groupby([col_grupo, 'genero'], observed=True).size().unstack(fill_value=0)
    ct['pct_feminino'] = ct['Feminino'] / (ct['Feminino'] + ct['Masculino']) * 100
    return ct['pct_feminino']


def prop_dentro_genero_regiao(df, col_grupo, regiao, genero):
    """
    Proporção de col_grupo dentro de um gênero específico, para uma região.
    Ex: de todas as mulheres do Sudeste, quantas % são Júnior?
    """
    sub = df[(df['regiao'] == regiao) & (df['genero'] == genero)].dropna(subset=[col_grupo])
    if sub.empty:
        return pd.Series(dtype=float)
    ct = sub[col_grupo].value_counts()
    return (ct / ct.sum() * 100)

---
## G-R1 — Representatividade Feminina por Região e Ano
> % de respondentes femininas em cada região ao longo dos 4 anos.

In [5]:
fig = go.Figure()

for regiao in REGIOES:
    pcts = []
    for ano in ANOS:
        sub = df_reg[(df_reg['regiao'] == regiao) & (df_reg['ano_pesquisa'] == ano)]
        if sub.empty:
            pcts.append(None)
            continue
        ct = sub['genero'].value_counts()
        fem = ct.get('Feminino', 0)
        total = ct.get('Feminino', 0) + ct.get('Masculino', 0)
        pcts.append(round(fem / total * 100, 1) if total > 0 else None)

    fig.add_trace(go.Scatter(
        x=ANOS, y=pcts,
        mode='lines+markers+text',
        name=regiao,
        line=dict(color=CORES_REGIAO[regiao], width=2.5),
        marker=dict(size=9),
        text=[f'{v}%' if v else '' for v in pcts],
        textposition='top center',
    ))

fig = layout_padrao(fig, '% de Respondentes Femininas por Região (2021–2024)', height=450)
fig.update_yaxes(ticksuffix='%', range=[0, 40], title_text='% Feminino')
fig.update_xaxes(title_text='Ano')
fig.show()

---
## G-R2 — Volume de Respondentes por Região e Ano
> Entender o peso amostral de cada região antes de comparar proporções.

In [6]:
vol = (
    df_reg.groupby(['ano_pesquisa', 'regiao'], observed=True)
    .size().reset_index(name='n')
)

fig = go.Figure()
for regiao in REGIOES:
    sub = vol[vol['regiao'] == regiao]
    fig.add_trace(go.Bar(
        x=sub['ano_pesquisa'],
        y=sub['n'],
        name=regiao,
        marker_color=CORES_REGIAO[regiao],
        text=sub['n'],
        textposition='auto',
    ))

fig = layout_padrao(fig, 'Volume de Respondentes por Região e Ano', height=430, barmode='group')
fig.update_yaxes(title_text='Respondentes')
fig.update_xaxes(title_text='Ano')
fig.show()

---
## G-R3 — Faixa Salarial por Região (acumulado)
> Proporção de cada faixa salarial dentro de cada região — permite comparar onde os salários são mais altos.

In [7]:
df_sal = df_reg.dropna(subset=['faixa_salarial'])

fig = make_subplots(
    rows=1, cols=len(REGIOES),
    subplot_titles=REGIOES,
    shared_yaxes=True
)

for i, regiao in enumerate(REGIOES, 1):
    sub = df_sal[df_sal['regiao'] == regiao]
    ct = sub['faixa_salarial'].value_counts().reindex(FAIXA_SALARIAL_ORDEM, fill_value=0)
    pct = (ct / ct.sum() * 100).round(1)

    fig.add_trace(go.Bar(
        y=FAIXA_SALARIAL_ORDEM,
        x=pct.values,
        name=regiao,
        orientation='h',
        marker_color=CORES_REGIAO[regiao],
        text=[f'{v:.0f}%' for v in pct.values],
        textposition='auto',
    ), row=1, col=i)
    fig.update_xaxes(ticksuffix='%', range=[0, pct.max() * 1.3], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, 'Distribuição de Faixa Salarial por Região — Acumulado 2021–2024',
                    height=550, width=1400)
fig.update_yaxes(title_text='Faixa Salarial', col=1)
fig.show()

---
## G-R4 — Faixa Salarial × Região × Gênero
> % feminino dentro de cada faixa salarial, comparado entre regiões. Revela onde a desigualdade salarial de gênero é mais ou menos acentuada.

In [8]:
df_sal = df_reg.dropna(subset=['faixa_salarial'])

fig = go.Figure()

for regiao in REGIOES:
    sub = df_sal[df_sal['regiao'] == regiao]
    pct = pct_feminino_por_grupo(sub, 'faixa_salarial').reindex(FAIXA_SALARIAL_ORDEM)

    fig.add_trace(go.Scatter(
        x=FAIXA_SALARIAL_ORDEM,
        y=pct.values,
        mode='lines+markers',
        name=regiao,
        line=dict(color=CORES_REGIAO[regiao], width=2),
        marker=dict(size=7),
    ))

# Linha de referência: % feminino geral
pct_geral = pct_feminino_por_grupo(df_sal, 'faixa_salarial').reindex(FAIXA_SALARIAL_ORDEM)
fig.add_trace(go.Scatter(
    x=FAIXA_SALARIAL_ORDEM,
    y=pct_geral.values,
    mode='lines',
    name='Brasil (geral)',
    line=dict(color='black', width=1.5, dash='dot'),
))

fig = layout_padrao(fig, '% Feminino por Faixa Salarial e Região — Acumulado 2021–2024', height=450)
fig.update_yaxes(ticksuffix='%', title_text='% Feminino na faixa')
fig.update_xaxes(tickangle=35, title_text='Faixa Salarial')
fig.show()

KeyError: 'Feminino'

---
## G-R5 — Pirâmide de Senioridade por Região
> Proporção dentro do gênero: de todas as mulheres de cada região, quantas % estão em cada nível?

In [ ]:
df_niv = df_reg.dropna(subset=['nivel'])

fig = make_subplots(
    rows=1, cols=len(REGIOES),
    subplot_titles=REGIOES,
    shared_yaxes=True
)

for i, regiao in enumerate(REGIOES, 1):
    sub = df_niv[df_niv['regiao'] == regiao]

    for genero in ['Feminino', 'Masculino']:
        pct = prop_dentro_genero_regiao(df_niv, 'nivel', regiao, genero)
        pct = pct.reindex(NIVEL_SENIORIDADE_ORDEM, fill_value=0)
        x_vals = -pct if genero == 'Feminino' else pct

        fig.add_trace(go.Bar(
            y=NIVEL_SENIORIDADE_ORDEM,
            x=x_vals,
            name=genero,
            orientation='h',
            marker_color=CORES_GENERO[genero],
            text=[f'{int(round(v))}%' for v in pct],
            textposition='outside',
        ), row=1, col=i)

    fig.add_shape(
        type='line', x0=0, x1=0,
        y0=-0.5, y1=len(NIVEL_SENIORIDADE_ORDEM) - 0.5,
        line=dict(color='black', width=1, dash='dot'),
        row=1, col=i
    )
    fig.update_xaxes(ticksuffix='%', range=[-80, 80], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(
    fig,
    'Pirâmide de Senioridade por Região e Gênero — Acumulado 2021–2024<br>'
    '<sub>Proporção dentro do gênero: de todas as mulheres/homens da região, quantas % estão em cada nível?</sub>',
    height=380, width=1400, barmode='relative'
)
fig.show()

---
## G-R6 — Escolaridade × Região × Gênero
> % feminino dentro de cada nível de ensino por região.

In [ ]:
df_esc = df_reg.dropna(subset=['nivel_ensino'])

fig = go.Figure()

for regiao in REGIOES:
    sub = df_esc[df_esc['regiao'] == regiao]
    pct = pct_feminino_por_grupo(sub, 'nivel_ensino').reindex(NIVEL_ENSINO_ORDEM)

    fig.add_trace(go.Scatter(
        x=NIVEL_ENSINO_ORDEM,
        y=pct.values,
        mode='lines+markers',
        name=regiao,
        line=dict(color=CORES_REGIAO[regiao], width=2),
        marker=dict(size=8),
    ))

# Referência geral
pct_geral = pct_feminino_por_grupo(df_esc, 'nivel_ensino').reindex(NIVEL_ENSINO_ORDEM)
fig.add_trace(go.Scatter(
    x=NIVEL_ENSINO_ORDEM, y=pct_geral.values,
    mode='lines', name='Brasil (geral)',
    line=dict(color='black', width=1.5, dash='dot'),
))

fig = layout_padrao(fig, '% Feminino por Nível de Escolaridade e Região — Acumulado 2021–2024', height=430)
fig.update_yaxes(ticksuffix='%', title_text='% Feminino no nível')
fig.update_xaxes(tickangle=20, title_text='Nível de Ensino')
fig.show()

---
## G-R7 — Cargo × Região × Gênero
> % feminino dentro de cada cargo, por região.

In [ ]:
df_car = df_reg[~df_reg['cargo'].isin(['Outro'])].dropna(subset=['cargo'])
cargos_ordem = df_car['cargo'].value_counts().index.tolist()

fig = make_subplots(
    rows=1, cols=len(REGIOES),
    subplot_titles=REGIOES,
    shared_yaxes=True
)

for i, regiao in enumerate(REGIOES, 1):
    sub = df_car[df_car['regiao'] == regiao]
    pct = pct_feminino_por_grupo(sub, 'cargo').reindex(cargos_ordem, fill_value=0)

    fig.add_trace(go.Bar(
        y=cargos_ordem,
        x=pct.values,
        name=regiao,
        orientation='h',
        marker_color=CORES_REGIAO[regiao],
        text=[f'{v:.0f}%' for v in pct.values],
        textposition='auto',
    ), row=1, col=i)
    fig.update_xaxes(ticksuffix='%', range=[0, 55], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, '% Feminino por Cargo e Região — Acumulado 2021–2024',
                    height=430, width=1400)
fig.update_yaxes(title_text='Cargo', col=1)
fig.show()

---
## G-R8 — Modalidade de Trabalho × Região × Gênero
> % feminino em cada modalidade (remoto/híbrido/presencial) por região.

In [ ]:
df_mod = df_reg.dropna(subset=['modalidade'])
modalidades_ordem = df_mod['modalidade'].value_counts().index.tolist()

fig = go.Figure()

for regiao in REGIOES:
    sub = df_mod[df_mod['regiao'] == regiao]
    pct = pct_feminino_por_grupo(sub, 'modalidade').reindex(modalidades_ordem, fill_value=0)

    fig.add_trace(go.Bar(
        x=modalidades_ordem,
        y=pct.values,
        name=regiao,
        marker_color=CORES_REGIAO[regiao],
        text=[f'{v:.0f}%' for v in pct.values],
        textposition='auto',
    ))

fig = layout_padrao(fig, '% Feminino por Modalidade de Trabalho e Região — Acumulado 2021–2024',
                    height=430, barmode='group')
fig.update_yaxes(ticksuffix='%', range=[0, 50], title_text='% Feminino na modalidade')
fig.update_xaxes(title_text='Modalidade')
fig.show()

---
## G-R9 — Situação de Trabalho × Região × Gênero
> % feminino em cada situação (CLT, PJ, Freelancer etc.) por região.

In [ ]:
df_sit = df_reg.dropna(subset=['situacao'])
situacoes_ordem = df_sit['situacao'].value_counts().index.tolist()

fig = make_subplots(
    rows=1, cols=len(REGIOES),
    subplot_titles=REGIOES,
    shared_yaxes=True
)

for i, regiao in enumerate(REGIOES, 1):
    sub = df_sit[df_sit['regiao'] == regiao]
    pct = pct_feminino_por_grupo(sub, 'situacao').reindex(situacoes_ordem, fill_value=0)

    fig.add_trace(go.Bar(
        y=situacoes_ordem,
        x=pct.values,
        name=regiao,
        orientation='h',
        marker_color=CORES_REGIAO[regiao],
        text=[f'{v:.0f}%' for v in pct.values],
        textposition='auto',
    ), row=1, col=i)
    fig.update_xaxes(ticksuffix='%', range=[0, 55], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, '% Feminino por Situação de Trabalho e Região — Acumulado 2021–2024',
                    height=480, width=1400)
fig.update_yaxes(title_text='Situação', col=1)
fig.show()

---
## G-R10 — Área de Formação × Região × Gênero
> % feminino em cada área de formação por região.

In [ ]:
df_area = df_reg.dropna(subset=['area_formacao'])
areas_ordem = df_area['area_formacao'].value_counts().index.tolist()

fig = make_subplots(
    rows=1, cols=len(REGIOES),
    subplot_titles=REGIOES,
    shared_yaxes=True
)

for i, regiao in enumerate(REGIOES, 1):
    sub = df_area[df_area['regiao'] == regiao]
    pct = pct_feminino_por_grupo(sub, 'area_formacao').reindex(areas_ordem, fill_value=0)

    fig.add_trace(go.Bar(
        y=areas_ordem,
        x=pct.values,
        name=regiao,
        orientation='h',
        marker_color=CORES_REGIAO[regiao],
        text=[f'{v:.0f}%' for v in pct.values],
        textposition='auto',
    ), row=1, col=i)
    fig.update_xaxes(ticksuffix='%', range=[0, 55], row=1, col=i)

fig = legenda_unica(fig)
fig = layout_padrao(fig, '% Feminino por Área de Formação e Região — Acumulado 2021–2024',
                    height=480, width=1400)
fig.update_yaxes(title_text='Área de Formação', col=1)
fig.show()

---
## G-R11 — Heatmap: % Feminino por Região × Ano
> Visão consolidada da evolução da representatividade feminina em cada região ao longo do tempo.

In [ ]:
matriz = pd.DataFrame(index=REGIOES, columns=ANOS, dtype=float)

for regiao in REGIOES:
    for ano in ANOS:
        sub = df_reg[(df_reg['regiao'] == regiao) & (df_reg['ano_pesquisa'] == ano)]
        ct = sub['genero'].value_counts()
        fem   = ct.get('Feminino', 0)
        total = ct.get('Feminino', 0) + ct.get('Masculino', 0)
        matriz.loc[regiao, ano] = round(fem / total * 100, 1) if total > 0 else np.nan

fig = go.Figure(go.Heatmap(
    z=matriz.values.astype(float),
    x=ANOS,
    y=REGIOES,
    text=[[f'{v:.1f}%' for v in row] for row in matriz.values.astype(float)],
    texttemplate='%{text}',
    colorscale='RdBu',
    zmid=20,          # centraliza a escala de cor em 20%
    zmin=0, zmax=40,
    colorbar=dict(title='% Feminino', ticksuffix='%'),
))

fig = layout_padrao(fig, 'Heatmap: % Feminino por Região e Ano', height=380)
fig.update_xaxes(title_text='Ano')
fig.update_yaxes(title_text='Região')
fig.show()